# DEIMv2 实时人员分析

本 Notebook 使用 DEIMv2 模型对视频进行实时人员检测和分析。

支持：
- 本地视频文件（如 test.mp4）
- 网络视频流地址（如 RTSP/HTTP 流）

功能：
- 实时人员检测与跟踪
- 人员数量统计
- 可视化展示

## 1. 导入依赖

In [ ]:
import torch
import torch.nn as nn
import cv2
import numpy as np
import colorsys
from PIL import Image, ImageDraw, ImageFont
from torchvision import transforms
from huggingface_hub import PyTorchModelHubMixin
from collections import deque
import time
from IPython.display import display, clear_output
import matplotlib.pyplot as plt

# 导入 DEIMv2 组件
from engine.backbone import HGNetv2, DINOv3STAs
from engine.deim import HybridEncoder, LiteEncoder
from engine.deim import DFINETransformer, DEIMTransformer
from engine.deim.postprocessor import PostProcessor

# warnings.filterwarnings('ignore')

# 配置 matplotlib 支持中文
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']  # 中文字体
plt.rcParams['axes.unicode_minus'] = False  # 解决负号显示问题

# 设备选择：自动检测 GPU，否则使用 CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch 版本: {torch.__version__}')
print(f'使用设备: {device}')
print("✅ 依赖导入成功")

## 2. 定义 DEIMv2 模型类

In [ ]:
class DEIMv2(nn.Module, PyTorchModelHubMixin):
    def __init__(self, config):
        super().__init__()
        if 'DINOv3STAs' in config:
            self.backbone = DINOv3STAs(**config["DINOv3STAs"])
        else:
            self.backbone = HGNetv2(**config["HGNetv2"])
        if 'LiteEncoder' in config:
            self.encoder = LiteEncoder(**config["LiteEncoder"])
        else:
            self.encoder = HybridEncoder(**config["HybridEncoder"])
        if 'DEIMTransformer' in config:
            self.decoder = DEIMTransformer(**config["DEIMTransformer"])
        else:
            self.decoder = DFINETransformer(**config["DFINETransformer"])
        self.postprocessor = PostProcessor(**config["PostProcessor"])

    def forward(self, x, orig_target_sizes):
        x = self.backbone(x)
        x = self.encoder(x)
        x = self.decoder(x)
        x = self.postprocessor(x, orig_target_sizes)
        return x

print("✅ DEIMv2 模型类定义完成")

## 3. 配置模型参数

选择合适的模型：
- **DEIMv2-S**: 推荐用于实时分析，AP 50.9，平衡速度和精度
- **DEIMv2-N**: 更快的速度，AP 43.0
- **DEIMv2-M/L/X**: 更高精度但速度较慢

In [ ]:
# DEIMv2-S 配置（与 hf_models.ipynb 完全一致）
deimv2_s_config = {
    "DINOv3STAs": {
        "name": "vit_tiny",
        "embed_dim": 192,
        "interaction_indexes": [5, 8, 11],
        "num_heads": 3
    },
    "HybridEncoder": {
        "in_channels": [192, 192, 192],
        "feat_strides": [8, 16, 32],
        "hidden_dim": 192,
        "use_encoder_idx": [2],
        "num_encoder_layers": 1,
        "nhead": 8,
        "dim_feedforward": 512,
        "dropout": 0.0,
        "enc_act": "gelu",
        "expansion": 0.34,
        "depth_mult": 0.67,
        "act": "silu",
        "version": "deim",
        "csp_type": "csp2",
        "fuse_op": "sum"
    },
    "DEIMTransformer": {
        "feat_channels": [192, 192, 192],
        "feat_strides": [8, 16, 32],
        "hidden_dim": 192,
        "num_levels": 3,
        "num_layers": 4,
        "eval_idx": -1,
        "num_queries": 300,
        "num_denoising": 100,
        "label_noise_ratio": 0.5,
        "box_noise_scale": 1.0,
        "reg_max": 32,
        "reg_scale": 4,
        "layer_scale": 1,
        "num_points": [3, 6, 3],
        "cross_attn_method": "default",
        "query_select_method": "default",
        "activation": "silu",
        "mlp_act": "silu",
        "dim_feedforward": 512,
        "eval_spatial_size": [640, 640]
    },
    "PostProcessor": {
        "num_top_queries": 300
    }
}

# COCO 类别映射（COCO 格式是 1-based）
# 注意：模型输出是 0-based 索引，使用时需要 +1 映射到 LABEL_MAP
LABEL_MAP = {
    1: 'person', 2: 'bicycle', 3: 'car', 4: 'motorcycle', 5: 'airplane',
    6: 'bus', 7: 'train', 8: 'truck', 9: 'boat', 10: 'traffic light',
    11: 'fire hydrant', 12: 'stop sign', 13: 'parking meter', 14: 'bench',
    15: 'bird', 16: 'cat', 17: 'dog', 18: 'horse', 19: 'sheep',
    20: 'cow', 21: 'elephant', 22: 'bear', 23: 'zebra', 24: 'giraffe',
    25: 'backpack', 26: 'umbrella', 27: 'handbag', 28: 'tie', 29: 'suitcase',
    30: 'frisbee', 31: 'skis', 32: 'snowboard', 33: 'sports ball', 34: 'kite',
    35: 'baseball bat', 36: 'baseball glove', 37: 'skateboard', 38: 'surfboard', 
    39: 'tennis racket', 40: 'bottle', 41: 'wine glass', 42: 'cup', 43: 'fork', 
    44: 'knife', 45: 'spoon', 46: 'bowl', 47: 'banana', 48: 'apple', 
    49: 'sandwich', 50: 'orange', 51: 'broccoli', 52: 'carrot', 53: 'hot dog', 
    54: 'pizza', 55: 'donut', 56: 'cake', 57: 'chair', 58: 'couch', 
    59: 'potted plant', 60: 'bed', 61: 'dining table', 62: 'toilet', 63: 'tv', 
    64: 'laptop', 65: 'mouse', 66: 'remote', 67: 'keyboard', 68: 'cell phone', 
    69: 'microwave', 70: 'oven', 71: 'toaster', 72: 'sink', 73: 'refrigerator', 
    74: 'book', 75: 'clock', 76: 'vase', 77: 'scissors', 78: 'teddy bear', 
    79: 'hair drier', 80: 'toothbrush'
}

# 人员检测配置
# 注意：模型输出是 0-based 索引，person 在 LABEL_MAP 中是第 1 类
# 所以模型输出的 0 对应 LABEL_MAP 的 1 (person)
PERSON_CLASS_ID = 0  # 模型输出的 0-based 索引
CONFIDENCE_THRESHOLD = 0.4
IMAGE_SIZE = (640, 640)

print("✅ 配置完成")
print(f"模型输入尺寸: {IMAGE_SIZE}")
print(f"置信度阈值: {CONFIDENCE_THRESHOLD}")
print(f"人员类别ID (模型输出 0-based): {PERSON_CLASS_ID}")
print(f"对应 LABEL_MAP 索引: {PERSON_CLASS_ID + 1} -> {LABEL_MAP[PERSON_CLASS_ID + 1]}")

## 4. 加载预训练模型

In [ ]:
# 选择设备
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"使用设备: {device}")

# 加载预训练模型
print("正在加载 DEIMv2-S 模型...")
model = DEIMv2.from_pretrained("Intellindust/DEIMv2_DINOv3_S_COCO")
model = model.to(device)
model.eval()

print("✅ 模型加载成功！")

## 5. 定义人员检测与分析类

In [ ]:
class PersonAnalyzer:
    """
    人员检测与分析类
    支持实时视频流和本地视频文件
    """
    
    def __init__(self, model, device, confidence_threshold=0.4, image_size=(640, 640)):
        self.model = model
        self.device = device
        self.confidence_threshold = confidence_threshold
        self.image_size = image_size
        
        # 图像预处理 - 参考 hf_models.ipynb，只使用 Resize 和 ToTensor
        self.transform = transforms.Compose([
            transforms.Resize(image_size),
            transforms.ToTensor(),
        ])
        
        # 统计信息
        self.person_count_history = deque(maxlen=100)
        self.fps_history = deque(maxlen=30)
        
    def detect_persons(self, image):
        """
        检测单帧图像中的人员
        
        Args:
            image: PIL.Image 或 numpy array
            
        Returns:
            detections: 检测到的人员列表 [{label, bbox, confidence}]
            person_count: 人员数量
        """
        if isinstance(image, np.ndarray):
            image = Image.fromarray(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
        
        orig_w, orig_h = image.size
        orig_size = torch.tensor([self.image_size]).to(self.device)
        
        # 预处理
        input_tensor = self.transform(image).unsqueeze(0).to(self.device)
        
        # 推理
        with torch.no_grad():
            outputs = self.model(input_tensor, orig_target_sizes=orig_size)
        
        # 参考 hf_models.ipynb 的输出格式
        labels = outputs[0]["labels"]
        boxes = outputs[0]["boxes"]
        scores = outputs[0]["scores"]
        
        # 筛选人员检测结果
        # 注意：模型输出是 0-based 索引，person 是第 0 类
        detections = []
        for label, bbox, score in zip(labels, boxes, scores):
            label_id = label.item()  # 0-based 索引
            score_val = score.item()
            
            # 只保留人员类别且置信度高于阈值
            if label_id == PERSON_CLASS_ID and score_val >= self.confidence_threshold:
                # bbox 转换为归一化格式 [x_min_ratio, y_min_ratio, width_ratio, height_ratio]
                # 与 hf_models.ipynb 保持一致
                detection = {
                    "label": "person",
                    "bbox": [
                        bbox[0].item() / self.image_size[0],  # x_min ratio
                        bbox[1].item() / self.image_size[1],  # y_min ratio
                        (bbox[2].item() - bbox[0].item()) / self.image_size[0],  # width ratio
                        (bbox[3].item() - bbox[1].item()) / self.image_size[1]   # height ratio
                    ],
                    "confidence": score_val,
                    "orig_size": (orig_w, orig_h)  # 保存原始尺寸用于绘制
                }
                detections.append(detection)
        
        return detections, len(detections)
    
    def draw_detections(self, image, detections, show_confidence=True):
        """
        在图像上绘制检测结果
        参考 hf_models.ipynb 的绘制方式
        """
        if isinstance(image, np.ndarray):
            image = Image.fromarray(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
        
        draw = ImageDraw.Draw(image)
        
        # 尝试加载字体
        try:
            font = ImageFont.truetype("arial.ttf", 20)
        except:
            font = ImageFont.load_default()
        
        for det in detections:
            bbox = det["bbox"]  # [x_min_ratio, y_min_ratio, width_ratio, height_ratio]
            confidence = det["confidence"]
            
            # 使用原始图片尺寸转换归一化坐标（与 hf_models.ipynb 一致）
            x_min = bbox[0] * image.width
            y_min = bbox[1] * image.height
            x_max = (bbox[0] + bbox[2]) * image.width
            y_max = (bbox[1] + bbox[3]) * image.height
            
            # 绘制边界框（绿色）
            draw.rectangle([x_min, y_min, x_max, y_max], outline="#00FF00", width=3)
            
            # 绘制标签
            if show_confidence:
                label_text = f"Person: {confidence:.2f}"
                # 文字背景
                draw.text((x_min, y_min - 13), label_text, fill="red", font=font)
        
        return image
    
    def process_video(self, video_source, output_path=None, max_frames=None, display_live=True):
        """
        处理视频（本地文件或网络流）
        
        Args:
            video_source: 视频文件路径或流地址
            output_path: 输出视频路径（可选）
            max_frames: 最大处理帧数（可选，用于测试）
            display_live: 是否实时显示（在 notebook 中）
        """
        # 打开视频
        cap = cv2.VideoCapture(video_source)
        
        if not cap.isOpened():
            print(f"❌ 无法打开视频源: {video_source}")
            return
        
        # 获取视频信息
        fps = cap.get(cv2.CAP_PROP_FPS)
        width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        
        print(f"\n📹 视频信息:")
        print(f"   分辨率: {width}x{height}")
        print(f"   FPS: {fps:.2f}")
        if total_frames > 0:
            print(f"   总帧数: {total_frames}")
        
        # 初始化视频写入器
        out = None
        if output_path:
            fourcc = cv2.VideoWriter_fourcc(*'mp4v')
            out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))
            print(f"   输出路径: {output_path}")
        
        frame_count = 0
        person_stats = {
            'total_frames': 0,
            'frames_with_persons': 0,
            'max_persons': 0,
            'avg_persons': 0,
            'person_count_sum': 0
        }
        
        print("\n🚀 开始处理视频...\n")
        
        try:
            while True:
                ret, frame = cap.read()
                if not ret:
                    break
                
                frame_count += 1
                
                # 检查是否达到最大帧数
                if max_frames and frame_count > max_frames:
                    break
                
                # 开始计时
                start_time = time.time()
                
                # 人员检测
                detections, person_count = self.detect_persons(frame)
                
                # 计算 FPS
                inference_time = time.time() - start_time
                current_fps = 1.0 / inference_time if inference_time > 0 else 0
                self.fps_history.append(current_fps)
                
                # 更新统计
                person_stats['total_frames'] += 1
                person_stats['person_count_sum'] += person_count
                if person_count > 0:
                    person_stats['frames_with_persons'] += 1
                person_stats['max_persons'] = max(person_stats['max_persons'], person_count)
                
                # 绘制结果
                result_frame = self.draw_detections(frame, detections)
                result_frame = cv2.cvtColor(np.array(result_frame), cv2.COLOR_RGB2BGR)
                
                # 添加统计信息覆盖
                avg_fps = np.mean(self.fps_history) if self.fps_history else 0
                self._add_overlay(result_frame, person_count, avg_fps, frame_count)
                
                # 保存到输出视频
                if out:
                    out.write(result_frame)
                
                # 实时显示（每 5 帧更新一次，避免卡顿）
                if display_live and frame_count % 5 == 0:
                    self._display_frame(result_frame, person_count, avg_fps)
                
                # 进度打印
                if frame_count % 30 == 0:
                    progress = f"Frame {frame_count}"
                    if total_frames > 0:
                        progress += f"/{total_frames} ({100*frame_count/total_frames:.1f}%)"
                    print(f"\r⏳ {progress} | 当前人数: {person_count} | FPS: {avg_fps:.1f}", end='')
                
        except KeyboardInterrupt:
            print("\n\n⚠️ 用户中断处理")
        
        finally:
            cap.release()
            if out:
                out.release()
            cv2.destroyAllWindows()
        
        # 计算最终统计
        if person_stats['total_frames'] > 0:
            person_stats['avg_persons'] = person_stats['person_count_sum'] / person_stats['total_frames']
        
        print(f"\n\n✅ 视频处理完成！")
        print(f"\n📊 统计结果:")
        print(f"   处理帧数: {person_stats['total_frames']}")
        print(f"   检测到人员的帧数: {person_stats['frames_with_persons']}")
        print(f"   最高人数: {person_stats['max_persons']}")
        print(f"   平均人数: {person_stats['avg_persons']:.2f}")
        
        return person_stats
    
    def _add_overlay(self, frame, person_count, fps, frame_num):
        """添加统计信息覆盖层"""
        # 半透明背景
        overlay = frame.copy()
        cv2.rectangle(overlay, (10, 10), (350, 120), (0, 0, 0), -1)
        cv2.addWeighted(overlay, 0.6, frame, 0.4, 0, frame)
        
        # 文字信息
        cv2.putText(frame, f"Persons: {person_count}", (20, 45),
                   cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
        cv2.putText(frame, f"FPS: {fps:.1f}", (20, 80),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 0), 2)
        cv2.putText(frame, f"Frame: {frame_num}", (20, 110),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.6, (200, 200, 200), 1)
    
    def _display_frame(self, frame, person_count, fps):
        """在 notebook 中显示当前帧"""
        clear_output(wait=True)
        plt.figure(figsize=(12, 8))
        plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        plt.axis('off')
        plt.title(f"人员检测 - 当前人数: {person_count}, FPS: {fps:.1f}")
        plt.tight_layout()
        plt.show()

print("✅ PersonAnalyzer 类定义完成")

## 6. 初始化分析器

In [ ]:
# 创建人员分析器实例
analyzer = PersonAnalyzer(
    model=model,
    device=device,
    confidence_threshold=0.4,
    image_size=(640, 640)
)

print("✅ 人员分析器初始化完成")

## 7. 处理本地视频文件

修改 `video_path` 为你的视频文件路径：

In [ ]:
# 本地视频文件路径
video_path = "xx.mp4"  # 修改为你的视频文件路径
output_path = "xx_person_analysis.mp4"

# 处理视频（处理前 300 帧作为示例，可根据需要调整或删除）
stats = analyzer.process_video(
    video_source=video_path,
    output_path=output_path,
    max_frames=50,  # 设置为 None 处理整个视频
    display_live=True  # 实时显示
)

## 8. 单帧测试

对单张图片进行人员检测（用于测试，与 hf_models.ipynb 完全一致）：

In [ ]:
# 单帧测试（与 hf_models.ipynb 完全一致）
test_image_path = "example.jpg"  # 替换为你的测试图片路径

import os
if os.path.exists(test_image_path):
    # 加载图片（与 hf_models.ipynb 一致）
    image = Image.open(test_image_path).convert("RGB")
    w, h = image.size
    print(f"图片尺寸: {w}x{h}")
    
    # 预处理（与 hf_models.ipynb 完全一致）
    transform = transforms.Compose([
        transforms.Resize(IMAGE_SIZE),
        transforms.ToTensor(),
    ])
    
    input_tensor = transform(image).unsqueeze(0).to(device)
    
    # 推理（与 hf_models.ipynb 完全一致）
    model.eval()
    with torch.no_grad():
        outputs = model(input_tensor, orig_target_sizes=torch.tensor([IMAGE_SIZE]))
    
    # 解析输出（与 hf_models.ipynb 一致）
    labels = outputs[0]["labels"]
    boxes = outputs[0]["boxes"]
    scores = outputs[0]["scores"]
    
    print(f"\n检测到 {len(labels)} 个目标")
    
    # 筛选人员检测结果（与 hf_models.ipynb 的 bbox 处理完全一致）
    # bbox 格式从 [x1, y1, x2, y2] 转换为归一化的 [x_min, y_min, width, height]
    detections = []
    for label, bbox, score in zip(labels, boxes, scores):
        label_id = label.item()  # 0-based
        score_val = score.item()
        coco_label_id = label_id + 1  # 转换为 1-based COCO ID
        
        print(f"  模型输出: label={label_id}, score={score_val:.3f}, bbox=[{bbox[0]:.1f}, {bbox[1]:.1f}, {bbox[2]:.1f}, {bbox[3]:.1f}]")
        
        # 筛选 person 类别（COCO ID = 1）
        if coco_label_id == 1 and score_val >= CONFIDENCE_THRESHOLD:  # person
            # 归一化 bbox 为 [x_min_ratio, y_min_ratio, width_ratio, height_ratio]
            detection = {
                "label": LABEL_MAP[coco_label_id],
                "bbox": [
                    bbox[0].item() / IMAGE_SIZE[0],  # x_min ratio
                    bbox[1].item() / IMAGE_SIZE[1],  # y_min ratio
                    (bbox[2].item() - bbox[0].item()) / IMAGE_SIZE[0],  # width ratio
                    (bbox[3].item() - bbox[1].item()) / IMAGE_SIZE[1]   # height ratio
                ],
                "confidence": score_val
            }
            detections.append(detection)
    
    print(f"\n✅ 筛选后人员数量: {len(detections)}")
    
    # 绘制结果（与 hf_models.ipynb 的绘制方式完全一致）
    draw = ImageDraw.Draw(image)
    try:
        font = ImageFont.truetype("arial.ttf", 20)
    except:
        font = ImageFont.load_default()
    
    for det in detections:
        label = det["label"]
        confidence = det["confidence"]
        bbox = det["bbox"]  # [x_min_ratio, y_min_ratio, width_ratio, height_ratio]
        
        # 使用原始图片尺寸转换归一化坐标
        x_min = bbox[0] * image.width
        y_min = bbox[1] * image.height
        x_max = (bbox[0] + bbox[2]) * image.width
        y_max = (bbox[1] + bbox[3]) * image.height
        
        # 绘制边界框（绿色）
        draw.rectangle([x_min, y_min, x_max, y_max], outline="#00FF00", width=3)
        # 绘制标签（红色文字，与 hf_models.ipynb 一致）
        draw.text((x_min, y_min - 13), f"{label}: {confidence:.2f}", fill="red", font=font)
    
    # 显示结果
    display(image)  # 使用 display 显示，与 notebook 环境更兼容
    
    print(f"\n最终结果: 检测到 {len(detections)} 个人员")
else:
    print(f"测试图片不存在: {test_image_path}")